In [ ]:
#---------------------------- IMPORTS --------------------------------------------#
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#---------------------------- LOAD DATA ------------------------------------------#
ratings = pd.read_csv('ml-latest-small/ratings.csv')
movies = pd.read_csv('ml-latest-small/movies.csv')
tags = pd.read_csv('ml-latest-small/tags.csv')

In [ ]:
#---------------------------- Data Explolration ----------------------------------#
print("============================== Ratings ============================================")
print(ratings.shape)                                        # how many rows/columms
print(ratings.head())                                       # first 5 rows
print(ratings.describe())                                   # min, max, mean rating etc.

print("\n============================ Movies =============================================")
print(movies.shape)
print(movies.head())

print("\n============================ Tags ===============================================")
print(tags.shape)
print(tags.head())

============================== Ratings ============================================
(100836, 4)
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931
              userId        movieId         rating     timestamp
count  100836.000000  100836.000000  100836.000000  1.008360e+05
mean      326.127564   19435.295718       3.501557  1.205946e+09
std       182.618491   35530.987199       1.042529  2.162610e+08
min         1.000000       1.000000       0.500000  8.281246e+08
25%       177.000000    1199.000000       3.000000  1.019124e+09
50%       325.000000    2991.000000       3.500000  1.186087e+09
75%       477.000000    8122.000000       4.000000  1.435994e+09
max       610.000000  193609.000000       5.000000  1.537799e+09

============================ Movies =============================================
(9742, 3

In [ ]:
#---------------------------- Building the Recommendation Model ------------------#

#---------------------------- IMPORTS --------------------------------------------#
from surprise import Dataset, Reader, SVD                    # for building the recommendation model
from surprise.model_selection import cross_validate          # for testing the model

# Rating scale
reader = Reader(rating_scale=(0.5, 5.0))

# Load the data from the DataFrame
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

# Build the SVD model
model = SVD(n_factors=150, n_epochs=30, lr_all=0.01, reg_all=0.1)

# Evaluate the model using cross-validation
results = cross_validate(model, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8605  0.8502  0.8551  0.8532  0.8565  0.8551  0.0034  
MAE (testset)     0.6574  0.6534  0.6559  0.6539  0.6580  0.6557  0.0018  
Fit time          0.74    0.81    0.81    0.73    0.74    0.77    0.04    
Test time         0.08    0.03    0.03    0.03    0.03    0.04    0.02    


In [ ]:
#---------------------------- IMPORTS --------------------------------------------#
from surprise import trainset
from collections import defaultdict

# Train on the whole dataset
trainset_full = data.build_full_trainset()
model.fit(trainset_full)

# Top N recommendations for a user
def get_recommendations(user_id, n=10):
    # Get list of all movie ids
    all_movie_ids = ratings['movieId'].unique()

    # Get movies the user already rated
    rated_movies = ratings[ratings['userId'] == user_id]['movieId'].tolist()

    # Predict ratings for ,ovies the user hasn't seen
    predictions = []
    for movie_id in all_movie_ids:
        if movie_id not in rated_movies:
            pred = model.predict(user_id, movie_id)
            predictions.append((movie_id, pred.est))

    # Sort by predicted rating and take top N
    predictions.sort(key=lambda x: x[1], reverse=True)
    top_n = predictions[:n]

    # Map movie ids to title
    results = []
    for movie_id, predicted_rating in top_n:
        title = movies[movies['movieId'] == movie_id]['title'].values[0]
        results.append((title, round(predicted_rating, 2)))

    return results
    
# Test is
user_id = 1
n = 10

recommendations = get_recommendations(user_id=user_id, n=n)
print(f"Top {n} recommendations for User {user_id}:")
for i, (title, score) in enumerate(recommendations, 1):
    print(f"{i}. {title} - \n   Predicted rating: {score}")

Top 10 recommendations for User 1:
1. Shawshank Redemption, The (1994) - 
   Predicted rating: 5.0
2. Wallace & Gromit: The Best of Aardman Animation (1996) - 
   Predicted rating: 5.0
3. Philadelphia Story, The (1940) - 
   Predicted rating: 5.0
4. Notorious (1946) - 
   Predicted rating: 5.0
5. Godfather, The (1972) - 
   Predicted rating: 5.0
6. General, The (1926) - 
   Predicted rating: 5.0
7. Creature Comforts (1989) - 
   Predicted rating: 5.0
8. Mary and Max (2009) - 
   Predicted rating: 5.0
9. Three Billboards Outside Ebbing, Missouri (2017) - 
   Predicted rating: 5.0
10. Ran (1985) - 
   Predicted rating: 5.0


In [ ]:
#---------------------------- Content-Based Filtering ----------------------------#

#---------------------------- IMPORTS --------------------------------------------#
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Clean up genres column - replace pipes with spaces
movies['genres_clean'] = movies['genres'].str.replace('|', ' ', regex=False)

# Build TF-IDF matrix
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(movies['genres_clean'])

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

# Compute cosine similarity between all movies
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(f"Cosine similarity matrix shape: {cosine_sim.shape}")

# Index mapping from movie titles to indices
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

# Function to get content-based recommendations
def get_content_recommendations(title, n=10):
    # Get the index of the movie
    idx = indices[title]

    # Get similarity scores for this movie with all others
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort by similarity score
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Ski[ index 0 (that's the same movie)
    sim_scores = sim_scores[1:n+1]

    # Get movie titles
    movie_indices = [i[0] for i in sim_scores]
    results = movies['title'].iloc[movie_indices].tolist()

    return results

# Test content-based recommendations
print("\nMovies similar to Toy Story (1995):")
for i, title in enumerate(get_content_recommendations("Toy Story (1995)"), 1):
    print(f"{i}. {title}")

print("\nMovies similar to The Dark Knight:")
for i, title in enumerate(get_content_recommendations("Dark Knight, The (2008)"), 1):
    print(f"{i}. {title}")


TF-IDF matrix shape: (9742, 24)
Cosine similarity matrix shape: (9742, 9742)

Movies similar to Toy Story (1995):
1. Antz (1998)
2. Toy Story 2 (1999)
3. Adventures of Rocky and Bullwinkle, The (2000)
4. Emperor's New Groove, The (2000)
5. Monsters, Inc. (2001)
6. Wild, The (2006)
7. Shrek the Third (2007)
8. Tale of Despereaux, The (2008)
9. Asterix and the Vikings (Astérix et les Vikings) (2006)
10. Turbo (2013)

Movies similar to The Dark Knight:
1. Need for Speed (2014)
2. Batman Begins (2005)
3. Fast Five (Fast and the Furious 5, The) (2011)
4. Eagle Eye (2008)
5. Good Day to Die Hard, A (2013)
6. Fast & Furious 6 (Fast and the Furious 6, The) (2013)
7. Grandmaster, The (Yi dai zong shi) (2013)
8. Dark Knight Rises, The (2012)
9. Man of Tai Chi (2013)
10. White House Down (2013)


In [17]:
#---------------------------- Hybrid Recommendation Function ---------------------#
def get_hybrid_recommendations(user_id, title, n=10, cf_weight=0.7, cb_weight=0.3):
    # Get content-based similarity scores for the given movie
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Get movies the user already rates 
    rated_movies = ratings[ratings['userId'] == user_id]['movieId'].tolist()

    # Combine CF and CB scores for rach movie
    hybrid_scores = []

    for movie_idx, cb_score in sim_scores:
        movie_id = movies.iloc[movie_idx]['movieId']
        movie_title = movies.iloc[movie_idx]['title']

        # Skip the movie itdelf
        if movie_title == title or movie_id in rated_movies:
            continue

        # Get CF predicted ratings for this user and movie
        cf_score = model.predict(user_id, movie_id).est

        # Normalize CF score from 0.5-5.0 range to 0-1 range
        cf_score_normalized = (cf_score - 0.5) / (5.0 - 0.5)

        # Calculate weighted hybrid score
        hybrid_score = (cf_weight * cf_score_normalized) + (cb_weight * cb_score)

        hybrid_scores.append((movie_title, hybrid_score, cf_score, cb_score))
    
    # Sort by hybrid score and return top N
    hybrid_scores.sort(key=lambda x: x[1], reverse=True)
    top_n = hybrid_scores[:n]

    return top_n

# Test hybrid recommendation
# User 1 based on Toy Story
print("Hybrid recommendations for User 1 based on 'Toy Story (1995)':")
print(f"{'Title':<50} {'Hybrid':>8} {'CF Score':>10} {'CB Score':>10}")
print("-" * 80)

recs = get_hybrid_recommendations(user_id=1, title="Toy Story (1995)", n=10)
for title, hybrid, cf, cb in recs:
    print(f"{title:<50} {hybrid:>8.4f} {cf:>10.4f} {cb:>10.4f}")

# User 15 based on Dark Knight
print("\nHybrid recommendations for User 15 based on 'Dark Knight, The (2008)':")
print(f"{'Title':<50} {'Hybrid':>8} {'CF Score':>10} {'CB Score':>10}")
print("-" * 80)

recs = get_hybrid_recommendations(user_id=15, title="Dark Knight, The (2008)", n=10)
for title, hybrid, cf, cb in recs:
    print(f"{title:<50} {hybrid:>8.4f} {cf:>10.4f} {cb:>10.4f}")

Hybrid recommendations for User 1 based on 'Toy Story (1995)':
Title                                                Hybrid   CF Score   CB Score
--------------------------------------------------------------------------------
Asterix and the Vikings (Astérix et les Vikings) (2006)   0.9633     4.7640     1.0000
Watership Down (1978)                                0.9601     4.8719     0.9333
Shrek (2001)                                         0.9436     4.7658     0.9333
Kiki's Delivery Service (Majo no takkyûbin) (1989)   0.9405     4.7462     0.9333
Monsters, Inc. (2001)                                0.9403     4.6159     1.0000
Spirited Away (Sen to Chihiro no kamikakushi) (2001)   0.9340     4.9218     0.8207
Ponyo (Gake no ue no Ponyo) (2008)                   0.9323     4.6349     0.9635
Partly Cloudy (2009)                                 0.9280     4.7125     0.9090
Toy Story 2 (1999)                                   0.9276     4.5347     1.0000
Toy Story 3 (2010)           